In [ ]:
pip install --upgrade scikit-learn

In [ ]:
pip install numpy==1.26.4 scikit-learn pandas

In [ ]:
pip install numpy==1.26.4 --force-reinstall

In [ ]:
print(f"NumPy version: {np.__version__}")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

In [2]:
import json
import ast
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import mlflow
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
import mlflow.pytorch
from tqdm import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import torch.nn as nn
from transformers import BertTokenizerFast
import sklearn
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

In [ ]:
directory_data = Path("/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset")
business_json = directory_data /"yelp_academic_dataset_business.json"
review_json = directory_data /"yelp_academic_dataset_review.json"
checkin_json = directory_data /"yelp_academic_dataset_checkin.json"

In [ ]:
df = pd.read_json(business_json, lines=True)
df.head()

In [ ]:
counts = df['is_open'].value_counts().sort_index()

plt.figure(figsize=(6,4))
sns.barplot(x=counts.index, y=counts.values, palette=['#d9534f', '#5cb85c'])
plt.xlabel('is_open (0 = закрыто, 1 = открыто)')
plt.ylabel('кол-во заведений')
plt.xticks([0,1])
plt.show()

In [ ]:
restaurants_df = df[df['categories'].str.contains('Restaurants', case=False, na=False)]
r_ids = set(restaurants_df['business_id'])

chunks = []
for ch in pd.read_json(review_json, lines=True, chunksize=100000):
    ch = ch[ch['business_id'].isin(r_ids)].copy()
    if not ch.empty:
        ch['date'] = pd.to_datetime(ch['date'])
        chunks.append(ch)

rev1 = pd.concat(chunks, ignore_index=True)
rev1.shape

In [ ]:
rev1 = rev1[['business_id', 'text', 'stars', 'date']]
df = df[['business_id', 'is_open']]

In [ ]:
df1 = rev1.merge(df, on='business_id', how='inner')
df1

In [ ]:
df1['date'] = pd.to_datetime(df1['date'])

last_review_date = df1.groupby('business_id')['date'].max().reset_index()
last_review_date.rename(columns={'date': 'last_review_date'}, inplace=True)

df1 = df1.merge(last_review_date, on='business_id', how='left')

In [ ]:
df1['cutoff_date'] = df1['last_review_date'] - pd.Timedelta(days=365)

df_filtered = df1[df1['date'] >= df1['cutoff_date']].copy()

In [ ]:
df_filtered.drop(columns=['cutoff_date', 'last_review_date'], inplace=True)

In [ ]:
df_filtered.shape

In [ ]:
df_filtered.to_csv('filtered_rew.csv', index=False, encoding='utf-8')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
rev1['text_len'] = rev1['text'].str.len() 
corr = rev1[['useful','funny','cool','text_len']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Корреляция между голосами и длиной текста')
plt.show()

In [ ]:
rev1['date'] = pd.to_datetime(rev1['date'])
rev1['year'] = rev1['date'].dt.year
rpy = rev1.groupby('year').size()
plt.figure(figsize=(12,6))
sns.lineplot(x=rpy.index, y=rpy.values, marker='o')
plt.title('Динамика количества отзывов по годам')
plt.xlabel('Год')
plt.ylabel('Количество отзывов')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
rev1['rating_group'] = '4-5 (положительные)'
rev1.loc[rev1['stars'] <= 3.5, 'rating_group'] = '1-3 (отрицательные)'

group_counts = rev1['rating_group'].value_counts().reindex(['1-3 (отрицательные)', '4-5 (положительные)'])

plt.figure(figsize=(8,6))
sns.barplot(x=group_counts.index, y=group_counts.values, palette=['red', 'green'])
plt.xlabel('Группа оценки')
plt.ylabel('Количество отзывов')
plt.show()

In [ ]:
rev.text.describe()

In [ ]:
rev1.drop_duplicates(subset='text', keep='first', inplace=True)

In [ ]:
rev1.shape

In [ ]:
rev1.to_csv('rev11.csv', index=False, encoding='utf-8')

In [ ]:
!pip install sweetviz

In [ ]:
# еще один инструмент автоматического EDA, я экспериментировала 
import sweetviz as sv
report = sv.analyze(rev1)
report.show_html()

In [ ]:
pip install torch==2.2.2 torchaudio==2.2.2 torchvision==0.17.2 torchtext==0.17.2 torchdata==0.11.0

In [ ]:
df_filtered.text.unique()

In [ ]:
df_filtered=pd.read_csv('filtered_rew.csv', encoding='utf-8')
df_filtered

In [ ]:
# приводим текст для токенизации
import re
df_filtered['text_clean'] = df_filtered['text'].str.lower().str.replace(r'[^a-z0-9\s\!\?]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
df_filtered.text_clean.unique()

In [3]:
df_final=pd.read_csv('/kaggle/input/datasets/fishshell/final-file/final (1).csv', encoding='utf-8')

In [4]:
review_counts = df_final['business_id'].value_counts()
valid_business_ids = review_counts[review_counts >= 10].index
df_final = df_final[df_final['business_id'].isin(valid_business_ids)].copy().reset_index()

In [5]:
df_final

,level_0,index,business_id,text,stars,date,is_open,text_clean,token_len,input_ids,attention_mask,label
0,0,0,uaipZDBSvzDzUUlazpyGCg,"The ramen is okay, nothing spectacular. Servic...",3,2016-01-04 07:26:09,0,the ramen is okay nothing spectacular service ...,34,"[101, 1996, 8223, 2368, 2003, 3100, 2498, 1265...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,1,1,Y2rYM2crYfTC25y6iquPgw,I stopped in for dinner and a drink & was thor...,5,2015-08-30 16:34:23,0,i stopped in for dinner and a drink was thorou...,105,"[101, 1045, 3030, 1999, 2005, 4596, 1998, 1037...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
2,2,3,EDjEVzmoQVHzboFqC-M6Ew,Lots of improvements to this place. Much more ...,5,2017-11-18 20:45:15,0,lots of improvements to this place much more c...,39,"[101, 7167, 1997, 8377, 2000, 2023, 2173, 2172...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,3,5,0Kn5W22UmxOqPj2cjouFNA,This is an updated review. I've been ordering ...,1,2015-10-08 04:32:10,0,this is an updated review i ve been ordering f...,182,"[101, 2023, 2003, 2019, 7172, 3319, 1045, 2310...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
4,4,7,i-tDq8zC7ZmSqSbg_7oddA,What an AMAZING occurrence that we ended up he...,5,2016-11-21 20:04:45,0,what an amazing occurrence that we ended up he...,342,"[101, 2054, 2019, 6429, 14404, 2008, 2057, 309...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
...,...,...,...,...,...,...,...,...,...,...,...,...
501694,501694,640346,c3QxX3toWdqJnKQmmIliRQ,"The supper club is ridiculously expensive. So,...",1,2021-11-29 18:26:40,1,the supper club is ridiculously expensive so w...,82,"[101, 1996, 15264, 2252, 2003, 9951, 2135, 645...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
501695,501695,640347,ReVpjIDupK_VMPn7ZxPvOQ,This place never fails the food is absolutely ...,4,2019-08-21 20:49:13,0,this place never fails the food is absolutely ...,66,"[101, 2023, 2173, 2196, 11896, 1996, 2833, 200...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
501696,501696,640348,uMVOtr16r1ELu46pWr4HCQ,Just average Thai food tonight. Bangkok has al...,1,2022-01-18 06:42:59,1,just average thai food tonight bangkok has alw...,181,"[101, 2074, 2779, 7273, 2833, 3892, 12627, 203...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
501697,501697,640350,e_E-jq9mwm7wk75k7Yi-Xw,It is very rare for a restaurant to be this go...,5,2022-01-17 22:36:01,1,it is very rare for a restaurant to be this go...,69,"[101, 2009, 2003, 2200, 4678, 2005, 1037, 4825...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1


In [12]:
df_final.token_len.max()

1178

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install numpy==1.26.4
!pip install transformers

In [6]:
# чтобы я могла видеть прогресс, а то я че то не вкуривала оно вообще работает или нет

# я применяю быструю BERT токенизацию
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

texts = df_final['text_clean'].astype(str).tolist()

batch_size = 1000 # поделила на батчи, чтобы быстрее работал, а то даже на 600к строках долго было
lengths = [] # для статистики длина токена

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    encodings = tokenizer(batch, add_special_tokens=True, truncation=False, padding=False)
    for ids in encodings['input_ids']:
        lengths.append(len(ids))

df_final['token_len'] = lengths

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

100%|██████████| 502/502 [01:44<00:00,  4.80it/s]


In [ ]:
print(tokenizer.save_pretrained(".")) # сохранила для гита на всякий

In [ ]:
df_final

In [7]:
# тут я уже создаю сами токены и маску для будущего обучения и кладу в датафрейм
ids_list = []
masks_list = []
max_len = int(df_final['token_len'].quantile(0.95)) # тк я не знаю какую длину взять, то возьмем такой перцентиль, чтобы не обрезать сильно слишком длинные отзывы

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encodings = tokenizer(batch, add_special_tokens=True, truncation=True, max_length=max_len, padding='max_length', return_tensors='pt')
    ids_list.append(encodings['input_ids'])
    masks_list.append(encodings['attention_mask'])

ids = torch.cat(ids_list, dim=0) # dim нужен, чтобы он склеил батчи
masks = torch.cat(masks_list, dim=0)

df_final['input_ids'] = ids.tolist()
df_final['attention_mask'] = masks.tolist()

In [ ]:
# сделаем тут колонку для разметки, типа 1 - хороший отзыв, 0 - плохой по кол-ву звезд 

df_final['label'] = (df_final['stars'] >= 4).astype(int)

# p.c оно не пригодится лол

In [13]:
# посмортрим наши метрики, может что интересное будет
metrics=df_final['token_len']

stats = {
    'count': len(metrics),
    'mean': metrics.mean(),
    'std': metrics.std(),
    'min': metrics.min(),
    '25%': metrics.quantile(0.25),
    '50%': metrics.quantile(0.50),
    '75%': metrics.quantile(0.75),
    '90%': metrics.quantile(0.90),
    '95%': metrics.quantile(0.95),
    '99%': metrics.quantile(0.99),
    'max': metrics.max()
}

metrics_df = pd.DataFrame([stats])
metrics_df

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
0,501699,105.903416,96.690082,2,43.0,76.0,134.0,218.0,289.0,486.0,1178


In [11]:
import torch.nn as nn
from transformers import BertTokenizerFast

# создаю слой, который буду обучать, он создает векторы для модели
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
vocab_size = tokenizer.vocab_size
embed_dim = 100 # берем 100, чтобы не расходовать сильно память, но получить норм результат
padding_idx = 0

embedding_layer = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx) # для моей ЛЛМ он не нужен, тк там есть встроенный

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
print(tokenizer.save_pretrained(".")) # сохранила для гита на всякий

('./tokenizer_config.json', './tokenizer.json')


In [10]:
# выделим теперь биграммы и триграммы, чтобы потом классифицировать, какие относятся к негативным отзывам, а какие к положительным
# они нам понадобятся для выявления проблемных фраз
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(2,3), max_features=100, stop_words='english')
X_ngrams = vectorizer.fit_transform(df_final['text_clean'])
feature_names = vectorizer.get_feature_names_out()

ValueError: np.nan is an invalid document, expected byte or unicode string.

In [ ]:
counts = np.asarray(X_ngrams.sum(axis=0)).flatten()
idx = np.argsort(counts)[-70:][::-1]
ngrams = feature_names[idx]
freq = counts[idx]

plt.figure(figsize=(10, 12))
plt.barh(ngrams[::-1], freq[::-1], color='skyblue')
plt.xlabel('Частота')
plt.title('Топ-70')
plt.tight_layout()
plt.show()

In [ ]:
pos_mask = (df_final['label'] == 1).values
neg_mask = (df_final['label'] == 0).values
X_arr = X_ngrams.toarray()

pos_counts = X_arr[pos_mask].sum(axis=0)[idx]
neg_counts = X_arr[neg_mask].sum(axis=0)[idx]

y_pos = np.arange(len(ngrams))
plt.figure(figsize=(10, 12))
plt.barh(y_pos - 0.2, pos_counts[::1], 0.4, label='Позитивные (label=1)', color='green')
plt.barh(y_pos + 0.2, neg_counts[::1], 0.4, label='Негативные (label=0)', color='red')
plt.yticks(y_pos, ngrams[::-1])
plt.xlabel('Частота')
plt.title('Топ-70')
plt.legend()
plt.show()

In [ ]:
# если честно он как-то не очень нашел биграммы и триграммы
# я не знаю почему((((
# надо как-то почистить их что ли.... но наверное не вручную
from scipy.sparse import csr_matrix

top5_ngrams = []
for i in range(X_ngrams.shape[0]):
    row = X_ngrams[i].toarray().flatten()
    present = [feature_names[j] for j in range(len(feature_names)) if row[j] > 0]
    top5_ngrams.append(present[:5])

df_final['top5_ngrams'] = top5_ngrams

In [4]:
df_final.to_csv('final_1.csv', encoding='utf-8',  index=False,)

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fishshell/final-file")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/fishshell/final-file


In [5]:
df_final=pd.read_csv('/kaggle/input/datasets/fishshell/final-file/final (1).csv', encoding='utf-8')

In [6]:
df_final.input_ids.astype

<bound method NDFrame.astype of 0         [101, 1996, 8223, 2368, 2003, 3100, 2498, 1265...
1         [101, 1045, 3030, 1999, 2005, 4596, 1998, 1037...
2         [101, 2023, 2173, 1055, 6866, 2847, 2421, 1037...
3         [101, 18040, 1055, 18651, 2003, 2026, 5440, 24...
4         [101, 1045, 2293, 2023, 4825, 1996, 5608, 2024...
                                ...                        
247144    [101, 2938, 2012, 1996, 3347, 2005, 2184, 2781...
247145    [101, 1996, 15264, 2252, 2003, 9951, 2135, 645...
247146    [101, 2074, 2779, 7273, 2833, 3892, 12627, 203...
247147    [101, 2009, 2003, 2200, 4678, 2005, 1037, 4825...
247148    [101, 2005, 2043, 1045, 1049, 3110, 2066, 9217...
Name: input_ids, Length: 247149, dtype: object>

In [ ]:
# я не могла скачать норм датасет отсюда, какой-то кринж лол
from IPython.display import FileLink
FileLink('filtered_after_token.csv')

In [11]:
pip install -q mlflow

Note: you may need to restart the kernel to use updated packages.


In [7]:
# из семинара
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [8]:
# это уже классы для обучения, почти все копипаст из доки и семинара

class RestaurantDataset(Dataset):
    def __init__(self, df):
        def parse_if_str(x):
            if isinstance(x, str):
                return ast.literal_eval(x)
            return x

        # тут я не могла понять, что не так с типами данных и перестраховалась, тк вечно ошибка возникала
        
        df['input_ids'] = df['input_ids'].apply(parse_if_str)
        df['attention_mask'] = df['attention_mask'].apply(parse_if_str)
    
        self.input_ids = torch.tensor(df['input_ids'].tolist(), dtype=torch.long)
        self.attention_mask = torch.tensor(df['attention_mask'].tolist(), dtype=torch.long)
        self.is_open = torch.tensor(df['is_open'].values, dtype=torch.long)

    def __len__(self):
        return len(self.is_open)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'is_open': self.is_open[idx]
        }
        
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        _, (hidden, _) = self.lstm(emb)
        hidden_last = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(self.dropout(hidden_last))

In [13]:
# гиперпараметры отдельно для будущих экспериментов
config = {
    "embed_dim": 100,
    "hidden_dim": 64,
    "num_lstm_layers": 1,
    "dropout": 0.5,
    "batch_size": 32,
    "learning_rate": 1e-3,
    "epochs": 5,
    "vocab_size": vocab_size_real,  
    "max_seq_len": 128,
    "num_classes": 2,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else 'cpu'
}
seed_everything(config['seed'])

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [12]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final['is_open'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['is_open'])

# берём из токенизатора
vocab_size_real = tokenizer.vocab_size  
print(f"Vocab size: {vocab_size_real}")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Vocab size: 30522
Train: 197719, Val: 24715, Test: 24715


In [16]:
from torch.utils.data import DataLoader
import ast

train_loader = DataLoader(
    RestaurantDataset(train_df),
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=(DEVICE.type == 'cuda')  
)

val_loader = DataLoader(
    RestaurantDataset(val_df),
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == 'cuda')
)

test_loader = DataLoader(
    RestaurantDataset(test_df),
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == 'cuda')
)

dataloaders = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_df), 'val': len(val_df)}
print(f"train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

train: 6179, val: 773, test: 773


In [17]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import torch
import torch.nn.functional as F

def train_model_bilstm(
    model: torch.nn.Module,
    criterion: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    dataloaders: dict,          
    dataset_sizes: dict,        
    device: torch.device,
    epochs: int = 5
) -> tuple[torch.nn.Module, dict]:
    history = {
        'train_loss': [], 'train_acc': [], 'train_prec': [], 'train_rec': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_prec': [], 'val_rec': [], 'val_f1': [], 'val_auc': []
    }
    best_model_weights = model.state_dict()
    best_accuracy = 0.0

    for epoch in range(epochs):
        print(f'Epoch {epoch+1}/{epochs}')
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            all_preds = []
            all_labels = []
            all_probs = []   
            for batch in tqdm(dataloaders[phase], desc=f'{phase.upper()}'):
                input_ids = batch['input_ids'].to(device)
                labels = batch['is_open'].to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(input_ids)
                    loss = criterion(logits, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * input_ids.size(0) 

                probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                preds = (probs >= 0.5).astype(int)

                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
                if phase == 'val':
                    all_probs.extend(probs)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = accuracy_score(all_labels, all_preds)
            prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)
            history[f'{phase}_prec'].append(prec)
            history[f'{phase}_rec'].append(rec)
            history[f'{phase}_f1'].append(f1)

            if phase == 'val':
                auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
                history['val_auc'].append(auc)
                print(f'Val Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Prec: {prec:.4f} Rec: {rec:.4f} F1: {f1:.4f} AUC: {auc:.4f}')
                if epoch_acc > best_accuracy:
                    best_accuracy = epoch_acc
                    best_model_weights = model.state_dict()
            else:
                print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Prec: {prec:.4f} Rec: {rec:.4f} F1: {f1:.4f}')  

    model.load_state_dict(best_model_weights)
    return model, history

In [52]:
import os

train_df.to_csv('/kaggle/working/.csv', index=False)

In [19]:
pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 91.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.9/936.9 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 16.3 MB/s eta 0:00:00
Note: you may need to restart t

In [20]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import torch.nn.functional as F

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for batch in tqdm(loader, desc="Train"):
        ids = batch['input_ids'].to(device)
        labels = batch['is_open'].to(device)         
        optimizer.zero_grad()
        logits = model(ids)                         
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    return avg_loss, acc, prec, rec, f1

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval"):
            ids = batch['input_ids'].to(device)
            labels = batch['is_open'].to(device)
            logits = model(ids)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return avg_loss, acc, prec, rec, f1, auc

In [21]:
import mlflow
import torch.nn as nn
from torch.utils.data import DataLoader
import os

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("bilstm_optimized")

run_name = (f"BiLSTM_emb{config['embed_dim']}_hid{config['hidden_dim']}_"
            f"layers{config['num_lstm_layers']}_lr{config['learning_rate']}_bs{config['batch_size']}")

with mlflow.start_run(run_name=run_name) as run:
    
    mlflow.log_params({k: v for k, v in config.items() if k != 'device'})
    
    # наша моделька
    model = BiLSTMClassifier(
        vocab_size=config['vocab_size'],
        embed_dim=config['embed_dim'],
        hidden_dim=config['hidden_dim'],
        num_layers=config['num_lstm_layers'],
        num_classes=config['num_classes'],
        dropout=config['dropout']
    ).to(config['device'])

    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    
    dataloaders = {'train': train_loader, 'val': val_loader}
    dataset_sizes = {'train': len(train_df), 'val': len(val_df)}
    
    trained_model, history = train_model_bilstm(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=None,          
        dataloaders=dataloaders,
        dataset_sizes=dataset_sizes,
        device=config['device'],
        epochs=config['epochs']
    )
    
    for epoch in range(1, config['epochs'] + 1):
        metrics = {
            "train_loss": history['train_loss'][epoch-1],
            "train_accuracy": history['train_acc'][epoch-1],
            "train_precision": history['train_prec'][epoch-1],
            "train_recall": history['train_rec'][epoch-1],
            "train_f1": history['train_f1'][epoch-1],
            "val_loss": history['val_loss'][epoch-1],
            "val_accuracy": history['val_acc'][epoch-1],
            "val_precision": history['val_prec'][epoch-1],
            "val_recall": history['val_rec'][epoch-1],
            "val_f1": history['val_f1'][epoch-1],
            "val_roc_auc": history['val_auc'][epoch-1]
        }
        mlflow.log_metrics(metrics, step=epoch)
        print(f"Epoch {epoch}: train_loss={metrics['train_loss']:.4f} train_f1={metrics['train_f1']:.4f} | "
              f"val_loss={metrics['val_loss']:.4f} val_f1={metrics['val_f1']:.4f} val_auc={metrics['val_roc_auc']:.4f}")
    
    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = eval_epoch(
        trained_model, test_loader, criterion, config['device']
    )
    mlflow.log_metrics({
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "test_f1": test_f1,
        "test_roc_auc": test_auc
    })
    
    mlflow.pytorch.log_model(trained_model, "bilstm_model")
    
    print(f"Run ID: {run.info.run_id}")
    print(f"Лучший val F1: {max(history['val_f1']):.4f}, тестовый F1: {test_f1:.4f}")

2026/06/16 09:15:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/16 09:15:45 INFO mlflow.store.db.utils: Updating database tables
2026/06/16 09:15:47 INFO mlflow.tracking.fluent: Experiment with name 'bilstm_optimized' does not exist. Creating a new experiment.


Epoch 1/5


TRAIN: 100%|██████████| 6179/6179 [00:34<00:00, 181.21it/s]


Train Loss: 0.5728 Acc: 0.7235 Prec: 0.7288 Rec: 0.9833 F1: 0.8371


VAL: 100%|██████████| 773/773 [00:02<00:00, 285.80it/s]


...

Train Loss: 0.4666 Acc: 0.7847 Prec: 0.7967 Rec: 0.9424 F1: 0.8635


VAL: 100%|██████████| 773/773 [00:02<00:00, 286.29it/s]


Val Loss: 0.5327 Acc: 0.7501 Prec: 0.7877 Rec: 0.8956 F1: 0.8382 AUC: 0.7302
Epoch 1: train_loss=0.5728 train_f1=0.8371 | val_loss=0.5488 val_f1=0.8434 val_auc=0.6858
Epoch 2: train_loss=0.5352 train_f1=0.8428 | val_loss=0.5274 val_f1=0.8467 val_auc=0.7149
Epoch 3: train_loss=0.5088 train_f1=0.8502 | val_loss=0.5235 val_f1=0.8490 val_auc=0.7265
Epoch 4: train_loss=0.4873 train_f1=0.8571 | val_loss=0.5320 val_f1=0.8482 val_auc=0.7303
Epoch 5: train_loss=0.4666 train_f1=0.8635 | val_loss=0.5327 val_f1=0.8382 val_auc=0.7302


Eval: 100%|██████████| 773/773 [00:02<00:00, 291.45it/s]
2026/06/16 09:18:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:18:58 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/16 09:18:58 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/16 09:19:16 WARNING mlflow.utils.requirements_utils: Found to

Run ID: a4eb396f43d6453eb3e001c68821b06e
Лучший val F1: 0.8490, тестовый F1: 0.8374


In [23]:

history_df = pd.DataFrame(history)
history_df.insert(0, 'epoch', range(1, len(history_df) + 1))
history_df = history_df.round(4)
history_df

,epoch,train_loss,train_acc,train_prec,train_rec,train_f1,val_loss,val_acc,val_prec,val_rec,val_f1,val_auc
0,1,0.5728,0.7235,0.7288,0.9833,0.8371,0.5488,0.7400,0.7466,0.9690,0.8434,0.6858
1,2,0.5352,0.7442,0.7580,0.9490,0.8428,0.5274,0.7499,0.7602,0.9554,0.8467,0.7149
2,3,0.5088,0.7602,0.7751,0.9413,0.8502,0.5235,0.7512,0.7561,0.9680,0.8490,0.7265
3,4,0.4873,0.7732,0.7866,0.9415,0.8571,0.5320,0.7566,0.7720,0.9413,0.8482,0.7303
4,5,0.4666,0.7847,0.7967,0.9424,0.8635,0.5327,0.7501,0.7877,0.8956,0.8382,0.7302


In [24]:
history_df.to_csv('training_history.csv', index=False)

In [25]:
!mlflow ui --backend-store-uri sqlite:///mlflow.db

Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/06/16 09:21:23 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/06/16 09:21:23 INFO:     Started parent process [332]
2026/06/16 09:21:31 INFO:     Started server process [337]
2026/06/16 09:21:31 INFO:     Waiting for application startup.
2026/06/16 09:21:31 INFO:     Application startup complete.
2026/06/16 09:21:31 INFO:     Started server process [338]
2026/06/16 09:21:31 INFO:     Waiting for application startup.
2026/06/16 09:21:31 INFO:     Application startup complete.
2026/06/16 09:21:31 INFO:     Started server process [335]
2026/06/16 09:21:31 INFO:     Waiting for application startup.
2026/06/16 09:21:31 INFO:     Application startup complete.
2026/06/16 09:21:31 INFO:     Started server proce